# MangaTR: أوراق مقارنة التنظيف

بياخد صفحات، يكشف النصوص، يعمل الماسك، وينضّف الصفحة بكذا طريقة، ويطلّع **ورقة لكل صفحة**: الأصل | الصناديق | الماسك | `opencv` | `aot` | `lama` | `cheap+lama` (الـ router المصمم)، وكل عمود عليه زمنه. وفي الآخر `timings.json` فيه الأزمنة وعدد الـ blocks اللي مرّ على كل طبقة.

**قبل ما تبدأ:** runtime **T4 GPU**، وسر `GITHUB_TOKEN` زي notebook الفحص (لو لسه ما أضفتوش شوف الـ notebook الأول).

الصفحات اللي هتتجرب: ارفعها في الخلية 4. جرّب 6 إلى 10 صفحات متنوعة (حبابة بيضاء، خلفية ملونة، نص فوق رسم، ويبتون طويل).

In [ ]:
# 1) الـ GPU
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "مفيش GPU")

In [ ]:
# 2) تحميل الكود وتثبيته
import os
import subprocess

from google.colab import userdata

BRANCH = "phase1/core-logic"
REPO_DIR = "/content/MangaTR"
PUBLIC_URL = "https://github.com/Nader-Geek/MangaTR.git"

if not os.path.isdir(REPO_DIR):
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = (
        PUBLIC_URL.replace("https://", f"https://x-access-token:{token}@") if token else PUBLIC_URL
    )
    done = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR],
        capture_output=True,
        text=True,
    )
    print((done.stdout + done.stderr).replace(token or "\0", "***")[-800:])
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", PUBLIC_URL])
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"])
%pip install -q -e "/content/MangaTR[gpu]" pytest

In [ ]:
# 3) الموديلات (لو نزلت قبل كده هيتخطى التحميل)
%cd /content/MangaTR
M = "/content/models"
!python -m mangatr.gpu_check --models-dir $M --runs 3 --cpu-runs 0 --out /content/r.json | tail -3
!ls -la $M

In [ ]:
# 4) ارفع الصفحات (png / jpg / webp). بتتحفظ في /content/pages
import os

from google.colab import files

os.makedirs("/content/pages", exist_ok=True)
for name, data in files.upload().items():
    with open(f"/content/pages/{name}", "wb") as handle:
        handle.write(data)
print(os.listdir("/content/pages"))

In [ ]:
# 5) التشغيل
%cd /content/MangaTR
P, M, S = "/content/pages", "/content/models", "/content/sheets"
!python -m mangatr.compare --images $P --models-dir $M --out $S

In [ ]:
# 6) اعرض الأوراق هنا (وتقدر تنزّلها من الخلية اللي بعدها)
import glob

from IPython.display import Image, display

for path in sorted(glob.glob("/content/sheets/*_sheet.jpg")):
    print(path)
    display(Image(path))

In [ ]:
# 7) نزّل الأوراق والأزمنة في ملف واحد
!cd /content && zip -qr sheets.zip sheets
from google.colab import files

files.download("/content/sheets.zip")

## ابعتلي إيه

1. **ملف `sheets.zip`** (أو لقطات الأوراق) ، وده الأهم لأن الجودة بتتحكم عليها بالعين.
2. محتوى `/content/sheets/timings.json` (الصقه في الشات).
3. لو خلية فشلت، آخر 15 سطر من خطأها (من غير التوكن).

الصفحات بتاعتك محفوظة على Colab بس ومش داخلة في git.